# 07 — Tourism Intensity

Quantifies tourism presence per census tract from OSM tourism-related POIs.

**Data source:** Overpass API (OSM `tourism=*`) — fully portable.

**Method:** Single batch Overpass query for all tourism POIs in the study area, then BallTree matching to assign each POI to its nearest tract.

**Output columns:** `tract_id`, `hotel_count`, `tourism_poi_count`, `tourism_density`, `tourism_ratio`

**Output file:** `csv/07_tourism_intensity.csv`

In [ ]:
ZONES_CONFIG = "zones.json"
QUERY_RADIUS = 500

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
import math
import hashlib
from sklearn.neighbors import BallTree

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

# Load amenity totals from notebook 02 for ratio computation
df_amenities = pd.read_csv("csv/02_amenity_composition.csv", dtype={"tract_id": str})
amenity_totals = dict(zip(df_amenities["tract_id"], df_amenities["amenity_count_total"]))

# Bounding box
BUFFER = 0.015
LAT_MIN = df_tracts["tract_lat"].min() - BUFFER
LAT_MAX = df_tracts["tract_lat"].max() + BUFFER
LON_MIN = df_tracts["tract_lon"].min() - BUFFER
LON_MAX = df_tracts["tract_lon"].max() + BUFFER
BBOX = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}

HOTEL_VALUES = {"hotel", "hostel", "motel", "guest_house"}
TOURISM_VALUES = {"hotel", "hostel", "motel", "guest_house", "museum",
                  "attraction", "viewpoint", "gallery", "artwork",
                  "information", "theme_park", "zoo", "aquarium"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            time.sleep(3 + attempt * 2)
    raise RuntimeError(f"Overpass failed: {last_error}")


print("Helpers ready.")

In [ ]:
# ── Batch query: ALL tourism POIs in study area ───────

query = (f'[out:json][timeout:90];\n'
         f'(node["tourism"]({BBOX});\n'
         f' way["tourism"]({BBOX}););\n'
         f'out center tags;')

print("Querying all tourism POIs...")
data = query_overpass_cached(query)

# Extract POIs with coordinates and tourism value
HOTEL_VALUES = {"hotel", "hostel", "motel", "guest_house"}
TOURISM_VALUES = {"hotel", "hostel", "motel", "guest_house", "museum",
                  "attraction", "viewpoint", "gallery", "artwork",
                  "information", "theme_park", "zoo", "aquarium"}

poi_records = []
for el in data.get("elements", []):
    tags = el.get("tags", {})
    tourism_val = tags.get("tourism", "")
    lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
    lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
    if lat and lon and tourism_val in TOURISM_VALUES:
        poi_records.append({
            "lat": float(lat), "lon": float(lon),
            "is_hotel": tourism_val in HOTEL_VALUES,
        })

print(f"  Found {len(poi_records)} tourism POIs ({sum(p['is_hotel'] for p in poi_records)} hotels)")

# ── BallTree: assign each POI to nearest tract ────────
EARTH_RADIUS_M = 6371000
AREA_KM2 = math.pi * (QUERY_RADIUS / 1000) ** 2
MAX_DIST_M = QUERY_RADIUS  # only count POIs within this distance of tract centroid

tract_coords_rad = np.radians(df_tracts[["tract_lat", "tract_lon"]].values)

# Initialize counts per tract
tract_ids = df_tracts["tract_id"].tolist()
hotel_counts = {t: 0 for t in tract_ids}
tourism_counts = {t: 0 for t in tract_ids}

if poi_records:
    tract_tree = BallTree(tract_coords_rad, metric="haversine")
    poi_coords = np.radians([[p["lat"], p["lon"]] for p in poi_records])
    distances, indices = tract_tree.query(poi_coords, k=1)
    
    for j, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        if dist * EARTH_RADIUS_M <= MAX_DIST_M:
            tid = tract_ids[idx]
            tourism_counts[tid] += 1
            if poi_records[j]["is_hotel"]:
                hotel_counts[tid] += 1

# Build result
records = []
for tid in tract_ids:
    total_amenities = amenity_totals.get(tid, 0)
    tc = tourism_counts[tid]
    records.append({
        "tract_id": tid,
        "hotel_count": hotel_counts[tid],
        "tourism_poi_count": tc,
        "tourism_density": round(tc / AREA_KM2, 2),
        "tourism_ratio": round(tc / total_amenities, 4) if total_amenities > 0 else 0.0,
    })

df_tourism = pd.DataFrame(records)
print(f"\nCompleted: {len(df_tourism)} tracts")
print(f"Tracts with hotels: {(df_tourism['hotel_count'] > 0).sum()}")
print(f"Tracts with tourism POIs: {(df_tourism['tourism_poi_count'] > 0).sum()}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/07_tourism_intensity.csv"
df_tourism.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_tourism)} rows x {df_tourism.shape[1]} cols)")
print(df_tourism.describe().round(2).to_string())